# XGBoost

## 1. Load Data

In [ ]:
import sys
sys.path.append('../src')
from data_prep import load_data

X_train, X_test, y_train, y_test, amount_test = load_data()

## 2. Establish Baseline

Hand-picked parameters, kept as a reference point for the tuning step below.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score
import matplotlib.pyplot as plt

ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_baseline = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio,  # ratio = (number of negatives / number of positives)
    eval_metric='logloss',
    random_state=42,
)
xgb_baseline.fit(X_train, y_train)

y_pred_baseline = xgb_baseline.predict(X_test)
y_prob_baseline = xgb_baseline.predict_proba(X_test)[:, 1]

print("Classification Report (hand-picked baseline params):")
print(classification_report(y_test, y_pred_baseline))
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob_baseline))

cm = confusion_matrix(y_test, y_pred_baseline)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraudulent'])
disp.plot(cmap='Purples')
plt.title("XGBoost Baseline: Fraud Detection Confusion Matrix")
plt.show()

## 3. Hyperparameter Tuning

Search for better parameters instead of hand-picking them, scored on PR-AUC (`average_precision`) since that's the right metric for this imbalance.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_distributions = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.3, 0.5],
}

search = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=ratio, eval_metric='logloss', random_state=42),
    param_distributions=param_distributions,
    n_iter=20,
    scoring='average_precision',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV PR-AUC:", search.best_score_)

xgb_model = search.best_estimator_

Only keep the tuned model if it actually beats the baseline under cross-validation — never decide this by peeking at the test set, since that would bias the "final" evaluation below.

In [ ]:
from sklearn.model_selection import cross_val_score

baseline_cv_score = cross_val_score(
    XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=ratio, eval_metric='logloss', random_state=42
    ),
    X_train, y_train, scoring='average_precision',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
).mean()

print(f"Baseline CV PR-AUC: {baseline_cv_score:.4f}")
print(f"Tuned CV PR-AUC:    {search.best_score_:.4f}")

if baseline_cv_score >= search.best_score_:
    print("\nBaseline performs at least as well under cross-validation -- keeping the baseline as the final model.")
    xgb_model = xgb_baseline
else:
    print("\nTuned model wins under cross-validation -- keeping the tuned model as the final model.")
    xgb_model = search.best_estimator_

## 4. PR-AUC & Threshold Tuning

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
import numpy as np

xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = xgb_model.predict(X_test)

# --- Default threshold (0.5), for reference ---
print("Classification Report (default threshold = 0.5, tuned model):")
print(classification_report(y_test, y_pred_xgb))

# --- PR-AUC: a better summary metric than ROC-AUC when classes are this imbalanced ---
pr_auc = average_precision_score(y_test, xgb_probs)
print(f"PR-AUC (Average Precision): {pr_auc:.4f}")

precision, recall, thresholds = precision_recall_curve(y_test, xgb_probs)

# --- Find the threshold that maximizes F1, instead of picking an arbitrary recall floor ---
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_f1_idx = np.argmax(f1_scores)
best_f1_threshold = thresholds[best_f1_idx]

print(f"\nF1-maximizing threshold: {best_f1_threshold:.4f}")
print(f"Precision: {precision[best_f1_idx]:.4f}  Recall: {recall[best_f1_idx]:.4f}  F1: {f1_scores[best_f1_idx]:.4f}")

y_pred_f1 = (xgb_probs >= best_f1_threshold).astype(int)
print("\nConfusion Matrix (F1-optimal threshold):")
print(confusion_matrix(y_test, y_pred_f1))
print("\nClassification Report (F1-optimal threshold):")
print(classification_report(y_test, y_pred_f1))

# --- Visualize the full precision-recall trade-off ---
plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label="PR curve")
plt.scatter(
    recall[best_f1_idx], precision[best_f1_idx],
    color="red", zorder=5, label=f"F1-optimal (t={best_f1_threshold:.3f})"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("XGBoost: Precision-Recall Curve")
plt.legend()
plt.show()

## 5. What if you specifically need higher recall?

Not a recommendation — just illustrating the precision cost of forcing a specific recall floor, e.g. if missed fraud has a known dollar cost that makes 90% recall a hard requirement.

In [ ]:
target_recall = 0.90
idxs = np.where(recall[:-1] >= target_recall)[0]  # drop last point (recall=0 has no threshold)

if len(idxs) > 0:
    best_idx = idxs[np.argmax(precision[idxs])]
    chosen_threshold = thresholds[best_idx]
    print(f"Chosen threshold: {chosen_threshold:.4f}")
    print(f"Precision at threshold: {precision[best_idx]:.4f}")
    print(f"Recall at threshold: {recall[best_idx]:.4f}")

    y_pred_tuned = (xgb_probs >= chosen_threshold).astype(int)
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred_tuned))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_tuned))
else:
    print("Model cannot reach that recall with meaningful thresholds.")

## 6. Cost-Based Threshold

Instead of a generic F1 balance, tie the threshold to real dollars: a missed fraud (false negative) costs its transaction `Amount`; a false alarm (false positive) costs a fixed review cost. Pick the threshold that minimizes total expected cost.

In [ ]:
REVIEW_COST = 10  # assumed dollar cost to investigate one flagged transaction -- adjust to your own estimate

y_test_arr = y_test.values
amount_arr = amount_test.values

costs = []
for t in thresholds:
    y_pred_t = (xgb_probs >= t).astype(int)
    fn_mask = (y_pred_t == 0) & (y_test_arr == 1)
    fp_mask = (y_pred_t == 1) & (y_test_arr == 0)
    cost = amount_arr[fn_mask].sum() + REVIEW_COST * fp_mask.sum()
    costs.append(cost)

costs = np.array(costs)
best_cost_idx = np.argmin(costs)
best_cost_threshold = thresholds[best_cost_idx]

print(f"Cost-minimizing threshold: {best_cost_threshold:.4f}")
print(f"Total cost at this threshold: ${costs[best_cost_idx]:,.2f}")
print(f"Precision: {precision[best_cost_idx]:.4f}  Recall: {recall[best_cost_idx]:.4f}")

y_pred_cost = (xgb_probs >= best_cost_threshold).astype(int)
print("\nConfusion Matrix (cost-optimal threshold):")
print(confusion_matrix(y_test, y_pred_cost))
print("\nClassification Report (cost-optimal threshold):")
print(classification_report(y_test, y_pred_cost))

plt.figure(figsize=(7, 5))
plt.plot(thresholds, costs)
plt.scatter(best_cost_threshold, costs[best_cost_idx], color="red", zorder=5, label=f"Min cost (t={best_cost_threshold:.3f})")
plt.xlabel("Threshold")
plt.ylabel("Total cost ($)")
plt.title("XGBoost: Cost vs. Threshold")
plt.legend()
plt.show()

## 7. Feature Importance

In [ ]:
import pandas as pd

importances = xgb_model.feature_importances_
feature_names = X_train.columns

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df)

## 8. Save Model & Predictions

So `06_model_comparison.ipynb` can load results without retraining.

In [ ]:
import json
import joblib

joblib.dump(xgb_model, '../models/xgb.pkl')
np.save('../predictions/xgb_probs.npy', xgb_probs)

with open('../models/xgb_thresholds.json', 'w') as f:
    json.dump({
        "f1_optimal_threshold": float(best_f1_threshold),
        "cost_optimal_threshold": float(best_cost_threshold),
        "review_cost_assumption": REVIEW_COST,
        "model_selected": "baseline" if xgb_model is xgb_baseline else "tuned",
        "baseline_cv_pr_auc": float(baseline_cv_score),
        "tuned_cv_pr_auc": float(search.best_score_),
        "tuned_params": search.best_params_,
    }, f, indent=2)